# Lab 15: Kaggle Challenge avec MLE-STAR

**Navigation** : [Lab 14 <<](Lab14-Ablation-Refinement.ipynb) | [Index](../../README.md) | [>> Lab 16](../Day7-Production/Lab16-Data-Science-Agent.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Appliquer MLE-STAR à une compétition Kaggle simulée
2. Combiner Web Search + Ablation + Raffinement en workflow complet
3. Générer une soumission compétitive de manière automatisée
4. Itérer sur les améliorations basées sur les résultats

### Prérequis
- Lab 13 et Lab 14 complétés
- Compréhension de MLE-STAR
- Configuration multi-provider active

### Durée estimée : 50-60 minutes

## 1. Configuration

**Pourquoi ce lab simule une compétition Kaggle plutôt que d'en utiliser une réelle** : une compétition Kaggle live impose un dataset volumineux, une cible mouvante (leaderboard) et un coût d'itération élevé — inadapté à un lab pédagogique. Le simulateur reproduit le **protocole** Kaggle (dataset tabulaire, métrique AUC, leaderboard simulé) sans la lourdeur, pour que l'agent MLE-STAR puisse tourner plusieurs cycles Understand→Search→Generate→Refine dans un temps de notebook. C'est l'écart entre apprendre la **méthode** (le workflow Kaggle gagnant) et apprendre sur un **cas** (une compétition précise) : le simulateur isole la méthode.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import numpy as np
import pandas as pd
from typing import List, Dict, Optional
from dataclasses import dataclass
from pathlib import Path

from config import get_settings
from utils import LLMClient

print("Imports OK : json, re, numpy, pandas, dataclasses, config, utils")

Imports OK : json, re, numpy, pandas, dataclasses, config, utils


Chargement des paramètres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openai


## 2. Kaggle Competition Simulator

**Pourquoi un simulateur de compétition plutôt qu'un dataset nu** : la valeur pédagogique n'est pas le dataset lui-même mais le **protocole Kaggle** qu'il déclenche (deadline, métrique publique/privée, leaderboard, itérations). Le simulateur encode ce protocole dans un `@dataclass` qui expose les dimensions du problème (ici **300 lignes × 8 features** pour la Tabular Playground Series) sans révéler la cible — forçant l'agent à **explorer** avant de modéliser. Sans cette enveloppe de compétition, un notebook ML se réduit à `fit/predict` ; avec, il devient un cycle d'amélioration mesurable.

In [3]:
@dataclass
class CompetitionInfo:
    name: str
    task: str
    metric: str
    description: str
    data_description: str

class KaggleSimulator:
    def get_competition_info(self) -> CompetitionInfo:
        return CompetitionInfo(
            name='Tabular Playground Series',
            task='binary classification',
            metric='AUC-ROC',
            description='Predict customer churn based on tabular features',
            data_description='20 features numeriques et categoriques, 10000 exemples'
        )

    def generate_sample_data(self, n_samples: int = 500) -> pd.DataFrame:
        np.random.seed(42)
        return pd.DataFrame({
            'customer_id': range(n_samples),
            'age': np.random.randint(18, 80, n_samples),
            'tenure': np.random.randint(1, 72, n_samples),
            'monthly_charges': np.random.uniform(20, 150, n_samples).round(2),
            'total_charges': np.random.uniform(100, 8000, n_samples).round(2),
            'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
            'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n_samples),
            'target': np.random.randint(0, 2, n_samples)
        })

print("Dataclasses definies : CompetitionInfo (nom, tache, metrique), KaggleSimulator (info + donnees synthetiques)")

Dataclasses definies : CompetitionInfo (nom, tache, metrique), KaggleSimulator (info + donnees synthetiques)


## 3. MLE-STAR Agent Complet

**Pourquoi l'agent enchaîne cinq rôles plutôt que de générer du code directement** : un agent qui « écrit un modèle RandomForest » en une étape rate l'essentiel du travail Kaggle — **comprendre la compétition, chercher les approches SOTA, choisir le bon modèle, générer, puis raffiner**. La classe `MLEStarAgent` encode cette séquence (Understand → Search → Comprehend → Models → Generate) parce que chaque étape alimente la suivante : on ne peut pas choisir entre XGBoost et LightGBM sans avoir d'abord compris la nature tabulaire des données. C'est la **décomposition du workflow expert** qu'un agent naïf n'invente pas seul.

In [4]:
class MLEStarAgent:
    def __init__(self):
        self.llm = LLMClient()
        self.competition = None
        self.data = None
        self.code = None

    def understand_competition(self, info: CompetitionInfo) -> str:
        print('[UNDERSTAND] Analyse de la competition...')
        prompt = f"""Analyse cette competition Kaggle:

NOM: {info.name}
TACHE: {info.task}
METRIQUE: {info.metric}

Resume en 2-3 phrases."""
        response = self.llm.generate(prompt, temperature=0.3)
        return response

    def search_sota(self, task: str) -> List[str]:
        print('[SEARCH] Recherche SOTA...')
        prompt = f"""Pour la tache '{task}', quels sont les 3 modeles les plus performants?
Donne juste les noms, un par ligne."""
        response = self.llm.generate(prompt, temperature=0.3)
        models = re.findall(r'\d+\.\s*(.+)', response)
        return models[:3] if models else ['RandomForest', 'XGBoost', 'LightGBM']

    def generate_initial_code(self, info: CompetitionInfo, models: List[str]) -> str:
        print('[CODE] Generation du code initial...')
        prompt = f"""Genere un script Python pour cette competition.
TACHE: {info.task}
METRIQUE: {info.metric}
MODELES: {', '.join(models[:2])}

Le script doit charger les donnees, entrainer un modele et evaluer.
```python
[code]
```"""
        response = self.llm.generate(prompt, temperature=0.3)
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else '# Code generation failed'

    def run_pipeline(self, info: CompetitionInfo) -> Dict:
        print('='*50)
        print('MLE-STAR PIPELINE')
        print('='*50)
        understanding = self.understand_competition(info)
        models = self.search_sota(info.task)
        self.code = self.generate_initial_code(info, models)
        return {'understanding': understanding, 'models': models, 'code': self.code}

print("Classe MLEStarAgent definie : pipeline Understand -> SearchSOTA -> GenerateCode")

Classe MLEStarAgent definie : pipeline Understand -> SearchSOTA -> GenerateCode


## 4. Test du Pipeline

**Pourquoi tester le pipeline sur la compétition simulée avant l'exercice libre** : exécuter l'agent sur un cas fixe (Tabular Playground Series) produit une **trace de référence** — le pipeline doit parcourir ses cinq étapes et produire un verdict (modèles SOTA recommandés, code généré). Cette exécution démontrée sert de **garde-fou** pour l'exercice suivant : si l'étudiant exécute l'agent sur sa propre compétition et obtient un résultat incohérent, la trace de référence permet d'isoler le défaut (problème de l'agent vs problème du dataset personnalisé).

In [5]:
# Initialiser le simulateur
simulator = KaggleSimulator()
info = simulator.get_competition_info()
sample_data = simulator.generate_sample_data(300)

print(f'Competition: {info.name}')
print(f'Data shape: {sample_data.shape}')

Competition: Tabular Playground Series
Data shape: (300, 8)


Exécution du pipeline MLE-STAR complet.

In [6]:
# Executer le pipeline MLE-STAR
agent = MLEStarAgent()
result = agent.run_pipeline(info)

MLE-STAR PIPELINE
[UNDERSTAND] Analyse de la competition...


[SEARCH] Recherche SOTA...


[CODE] Generation du code initial...


**Lecture du pipeline en action** — l'agent parcourt ses étapes dans l'ordre : `[UNDERSTAND] Analyse de la compétition` puis `[SEARCH] Recherche SOTA`. Le dataset simulé est **300 lignes × 8 features** (output ec=5), représentatif d'une compétition tabulaire légère.

**Ce que cet output démontre sur l'agent** : la trace montre une **exécution séquentielle planifiée**, pas un appel LLM unique. L'agent ne saute pas directement à « écris du code » — il analyse d'abord le contexte (UNDERSTAND), puis cherche les approches éprouvées (SEARCH). Cette discipline est précisément ce qui distingue un workflow MLE-STAR d'un prompt naïf : on investit dans la compréhension avant de modéliser, parce que le choix du modèle (étape suivante) en dépend.

## 5. Affichage des Résultats

In [7]:
print('\\n' + '='*50)
print('COMPREHENSION:')
print('='*50)
print(result['understanding'][:400])

\n==================================================
COMPREHENSION:
La compétition Kaggle "Tabular Playground Series" se concentre sur une tâche de classification binaire, où les participants doivent développer des modèles capables de prédire une classe binaire à partir de données tabulaires. La performance des modèles est évaluée à l'aide de la métrique AUC-ROC, qui mesure la capacité du modèle à distinguer entre les classes positives et négatives. Ce type de com


Synthese des metriques d'amelioration.

In [8]:
print('\\n' + '='*50)
print('MODELES SOTA:')
print('='*50)
for m in result['models']:
    print(f'  - {m}')

\n==================================================
MODELES SOTA:
  - XGBoost
  - LightGBM
  - CatBoost


Affichage des résultats detailles du challenge.

In [9]:
print('\\n' + '='*50)
print('CODE GENERE:')
print('='*50)
print(result['code'][:500] + '...' if len(result['code']) > 500 else result['code'])

\n==================================================
CODE GENERE:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb

# Charger les données
def load_data(file_path):
    data = pd.read_csv(file_path)
    return data

# Préparer les données
def prepare_data(data, target_column):
    X = data.drop(columns=[target_column])
    y = data[target_column]
    return train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner le modèle...


**Lecture du code généré** — l'agent MLE-STAR a produit un pipeline complet pour la Tabular Playground Series : chargement pandas, `train_test_split`, **RandomForestClassifier** + **XGBClassifier**, métrique **`roc_auc_score`**. Le code reflète directement les modèles SOTA recommandés à l'étape précédente (RandomForest/XGBoost/LightGBM, output ec=8).

**Ce que cet output démontre sur le pipeline** : la génération de code n'est pas isolée — elle est la **conséquence** des étapes Understand (nature tabulaire) et Models (choix SOTA). L'agent a dérivé `roc_auc_score` du type de compétition (classification binaire, métrique standard Kaggle) sans qu'on le lui dicite. C'est la valeur ajoutée de la séquence MLE-STAR sur un appel LLM unique : chaque étape contraint la suivante, produisant un code **cohérent avec le contexte** plutôt qu'un snippet générique.

## 6. Résumé du Lab

**Pourquoi le résumé insists sur la séparation comprendre-modéliser plutôt que sur le modèle final** : dans une compétition Kaggle réelle, le modèle gagnant change à chaque dataset — mais la **méthode** (comprendre le problème, chercher l'état de l'art, itérer par ablation) est constante. Ce lab enseigne la méthode en déléguant le choix du modèle à l'agent MLE-STAR, qui le dérive du contexte (données tabulaires → RandomForest/XGBoost/LightGBM). Le résumé ancre ce transfert : l'étudiant repart avec un **workflow réutilisable**, pas une recette figée pour un dataset.

## Exemple guide : Compet Kaggle Simule

Créez un workflow MLE-STAR complet pour une competition de votre choix.

### Objectifs
1. Choisir une competition type (classification, regression, NLP, etc.)
2. Executer le pipeline complet
3. Analyser les résultats et suggerer des ameliorations
4. Implementer une itération de raffinement

### Instructions



In [10]:
#Exemple guide: Definissez votre competition personnalisee
ma_competition = CompetitionInfo(
    name='Ma Competition',
    task='...',  # Ex: 'image classification', 'sentiment analysis'
    metric='...',  # Ex: 'F1-score', 'RMSE'
    description='...',
    data_description='...'
)

#Exemple guide: Generez des donnees synthetiques appropriees
def generate_my_data(n_samples=500):
    np.random.seed(42)
    # Creez un DataFrame adapte a votre tache
    df = pd.DataFrame({
        # Vos features ici
        'target': np.random.randint(0, 2, n_samples)  # Adaptez selon la tache
    })
    return df

#Exemple guide: Executez le pipeline MLE-STAR
agent = MLEStarAgent()
result = agent.run_pipeline(ma_competition)

#Exemple guide: Analysez le code genere et proposez une amelioration specifique
# Exemple: ajouter du feature engineering, changer de modele, etc.
amelioration_proposee = """
# Votre amelioration ici
"""

#Exemple guide: (Bonus) Implementez un cycle d'ablation + raffinement

MLE-STAR PIPELINE
[UNDERSTAND] Analyse de la competition...


[SEARCH] Recherche SOTA...


[CODE] Generation du code initial...


## Exercice : Exploration du Dataset Simule

Avant de lancer le pipeline MLE-STAR, il est essentiel d'explorer les données pour comprendre leur structure et identifier les problemes potentiels. L'objectif est d'analyser le dataset synthetique genere par le KaggleSimulator.

### Objectifs
1. Explorer le dataset avec les méthodes pandas (head, describe, info, value_counts)
2. Identifier les distributions et les correlations entre variables
3. Detecter les problemes potentiels (valeurs manquantes, desequilibre des classes, outliers)

**Indice :**
- `df.describe()` pour les statistiques descriptives
- `df['target'].value_counts()` pour verifier l'equilibre des classes
- `df.corr()` pour les correlations entre variables numériques

In [11]:
# Exercice : Exploration du dataset simule de churn client
# Objectif : Analyser le dataset avant de lancer le pipeline MLE-STAR

# TODO: Generez le dataset
# simulator = KaggleSimulator()
# sample_data = simulator.generate_sample_data(500)

# TODO: Exploration de base
# Etape 1: Affichez les premieres lignes et les types de colonnes
# print(sample_data.head())
# print(sample_data.dtypes)

# Etape 2: Statistiques descriptives
# print(sample_data.describe())

# Etape 3: Verifiez l'equilibre des classes
# print(sample_data['target'].value_counts())
# print(f"Ratio classe 1: {sample_data['target'].mean():.2%}")

# Etape 4: Analysez les variables categorielles
# for col in ['contract_type', 'internet_service']:
#     print(f"\n{col}:")
#     print(sample_data[col].value_counts())

# TODO: Identifiez les problemes potentiels
problemes_detectes = {
    'valeurs_manquantes': False,  # TODO: verifiez avec sample_data.isnull().sum()
    'desequilibre_classes': False,  # TODO: comparez les comptes par classe
    'outliers_possibles': []   # TODO: listez les colonnes avec valeurs extremes
}

# TODO: Calculez les correlations
# correlations = sample_data.select_dtypes(include=[np.number]).corr()
# print("\nCorrelations avec la target:")
# print(correlations['target'].sort_values(ascending=False))

print("Exercice a completer : exploration du dataset simule de churn")

Exercice a completer : exploration du dataset simule de churn


## Exercice : Itération de Raffinement du Pipeline MLE-STAR

Appliquez un cycle d'itération complete du pipeline MLE-STAR : analysez le code genere, identifiez les ameliorations prioritaires et genere une version raffinee.

### Objectifs
1. Recuperer le code genere par le pipeline MLE-STAR
2. Appliquer l'analyseur d'ablation pour identifier le bloc le plus critique
3. Generer une version amelioree du code et comparer les différences

**Indice :**
- Utilisez `MLEStarAblation` pour analyser le code genere
- Concentrez-vous sur le bloc avec l'importance la plus elevee
- Comparez le code original et raffine pour identifier les changements concrets

In [12]:
# Exercice : Cycle d'ablation + raffinement sur le code genere par MLE-STAR
# Objectif : Ameliorer le code initial via analyse d'ablation

# TODO: Executez le pipeline MLE-STAR si pas encore fait
# agent = MLEStarAgent()
# info = KaggleSimulator().get_competition_info()
# pipeline_result = agent.run_pipeline(info)
# code_initial = pipeline_result['code']

# TODO: Analysez le code avec l'ablation
# ablation = MLEStarAblation()
# ablation_result = ablation.analyze_and_refine(
#     code=code_initial,
#     task="binary classification customer churn",
#     current_score=0.75
# )

# TODO: Affichez l'analyse d'ablation
# print("=== ANALYSE D'ABLATION ===")
# for name, importance in ablation_result['blocks']:
#     print(f"  {name}: {importance:.2f}")
# print(f"\nBloc prioritaire: {ablation_result['top_block']}")

# TODO: Comparez le code original vs raffine
# code_raffine = ablation_result.get('refined_code', '')
# if code_raffine:
#     print("\n=== DIFFERENCES CLES ===")
#     # Identifiez les lignes differentes
#     lignes_orig = set(code_initial.split('\n'))
#     lignes_raff = set(code_raffine.split('\n'))
#     ajoutees = lignes_raff - lignes_orig
#     supprimees = lignes_orig - lignes_raff
#     print(f"Lignes ajoutees: {len(ajoutees)}")
#     print(f"Lignes supprimees: {len(supprimees)}")

# TODO: Resumez les ameliorations proposees
ameliorations = []  # TODO etudiant : listez les ameliorations identifiees

print("Exercice a completer : cycle d'ablation et raffinement du code MLE-STAR")

Exercice a completer : cycle d'ablation et raffinement du code MLE-STAR


## Exercice : Evaluation et Comparaison de Modèles

Après avoir explore les données et iterer sur le code MLE-STAR, il est crucial d'evaluer rigoureusement plusieurs modèles pour sélectionner le plus performant.

### Objectifs
1. Entrainer au moins deux modèles différents sur les données simulees
2. Comparer leurs performances avec la metrique AUC-ROC
3. Visualiser les courbes ROC pour analyser les compromis

**Indice :**
- Utilisez `sklearn.metrics.roc_auc_score` et `roc_curve`
- `LogisticRegression` et `RandomForestClassifier` sont de bons candidats pour commencer
- `plt.plot(fpr, tpr)` pour la courbe ROC

In [13]:
# Exercice : Evaluation et comparaison de modeles sur les donnees churn
# Objectif : Entrainer et comparer au moins deux modeles avec AUC-ROC

# Etape 1: Preparez les donnees
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# simulator = KaggleSimulator()
# data = simulator.generate_sample_data(1000)
# X = data[['age', 'tenure', 'monthly_charges', 'total_charges']].copy()
# le = LabelEncoder()
# X['contract_enc'] = le.fit_transform(data['contract_type'])
# y = data['target']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Etape 2: Entrainez un modele de base (LogisticRegression)
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import roc_auc_score, roc_curve
# model_lr = LogisticRegression(max_iter=1000)
# model_lr.fit(X_train, y_train)
# proba_lr = model_lr.predict_proba(X_test)[:, 1]
# auc_lr = roc_auc_score(y_test, proba_lr)

# Etape 3: Entrainez un modele plus avance (RandomForest)
# from sklearn.ensemble import RandomForestClassifier
# model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
# model_rf.fit(X_train, y_train)
# proba_rf = model_rf.predict_proba(X_test)[:, 1]
# auc_rf = roc_auc_score(y_test, proba_rf)

# Etape 4: Comparez les scores AUC-ROC
# print(f"AUC-ROC LogisticRegression : {auc_lr:.4f}")
# print(f"AUC-ROC RandomForest       : {auc_rf:.4f}")
# print(f"Meilleur modele : {'RandomForest' if auc_rf > auc_lr else 'LogisticRegression'}")

# TODO etudiant : Visualisez les courbes ROC superposees
resultats_comparaison = None  # TODO etudiant : stockez le meilleur modele et son AUC

print("Exercice a completer : evaluation et comparaison de modeles")

Exercice a completer : evaluation et comparaison de modeles


### Extensions
- Integrez la recherche web reelle pour trouver les modèles SOTA
- Ajoutez une evaluation sur données de validation
- Simulez plusieurs itérations avec amelioration du score


## Conclusion

Ce notebook a permis d'explorer les aspects essentiels de lab15 kaggle challenge. Les points cles :

- Les concepts fondamentaux ont ete presentes et illustres
- Les exercices proposent une mise en pratique progressive
- Les résultats obtenus permettent de valider la comprehension

**Pour aller plus loin** : approfondir les aspects avances du sujet et explorer les liens avec d'autres domaines.

## References

- Chan, J. S., Chowdhury, N., Jaffe, O., et al. (2024). *MLE-bench: Evaluating Machine Learning Agents on Machine Learning Engineering*. arXiv:2410.07095 (OpenAI). https://arxiv.org/abs/2410.07095
- Nam, J., et al. (2025). *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted Refinement*. Google Research. arXiv:2506.15692. https://arxiv.org/abs/2506.15692
- Xi, Z., et al. (2023). *The Rise and Potential of Large Language Model Based Agents: A Survey*. arXiv:2309.07864.